# **CMSC 320 - FINAL PROJECT**

In [ ]:
#imports
import pandas as pd
import string
import math
import matplotlib.pyplot as plt
from scipy import stats


**PART 1 - DATA PROCESSING**

We are using Earthquake data from the USGS earthquake data website.  This data is from June 1st, 2005 to September 27th, 2012. It includes not only earthquake data but data involving Volcano Eruption and Landsides all of which has been reviewd in the context of Earthquakes.

In [ ]:
earthquakes_df = pd.read_csv("Earthquakes.csv")
earthquakes_df

Information about the dataset before any modifications take place

In [ ]:
print(f"df.shape: {earthquakes_df.shape}")
earthquakes_df.shape
print("df.describe(): ")
display(earthquakes_df.describe())
print("df.info(): ")
earthquakes_df.info()

Parse data: Sort by magnitude

In [ ]:
#sort by mag highest to smallest
earthquakes_df = earthquakes_df.sort_values(by=['mag'], ascending=False).reset_index(drop='True')

#this line below is to check if there was mag error, there is, maybe we could use it
#earthquakes_df = earthquakes_df.dropna(subset=['magError'])

#date only column, in datetime (remove .dt.date to get full date and time in datetime format)
date = pd.to_datetime(earthquakes_df['time']).dt.date
earthquakes_df.insert(0, 'date', date)
display(earthquakes_df)

**PART 2**

Frequency of Earthquakes Based on Geographical Region
Method 2: Categorical distribution analysis

We will first examine the frequency of earthquakes based off of geographical region. To better visualize the correlation between how often the earthquakes happen and which locations get the most number of earthquakes, we will use a bar chart and a pie chart. First, we want to see how frequent are the earthquakes in the locations in our database. We notice that most of the locations have only 1 earthquake, with the graph being extremely right skewed that is almost imposible to be seen on the graph. From this first graph we understand that most of the regions in our data base have less than 20 earthquakes.

In [ ]:
place_types = earthquakes_df['place'].value_counts()

bins = [0,1, 5, 10, 20, float('inf')]
labels = ['1','2-5','6-10', '11-20', '21+']

binned = pd.cut(place_types, bins = bins, labels=labels)
binned_occurences = binned.value_counts().sort_index()

binned_occurences.plot(kind='bar')
plt.title('How often do earthquakes happen in our locations')
plt.xlabel('Number of earthquakes')
plt.ylabel('Number of locations')

plt.show()

We wanted to take a closer look at the locations that have more than 10 earthquakes. We wanted to see exactly which locations are prone to having more earthquakes and their exact number of earthquakes. We notice that there are 2 locations with an astonishing amount of earthquakes, more than 100. These 2 locations are the outliers that are skewing the graph. This made us wonder what is the distribution of the frequency of earthquakes.

In [ ]:
place_types_more_than_50_occurences = place_types[place_types > 10]

place_types_more_than_50_occurences.sort_values().plot(kind='bar')

plt.title("Locations with more than 10 earthquakes")
plt.ylabel('Number of earthquakes')
plt.xlabel('Locations')
plt.show()

Among out dataset, we wondered what is the percentage of locations that suffered 1 earthquake, between 2 and 10, and finally, more than 11. For this question we used the pie chart which perfectly shows that almost 88% of the locations have suffered only 1 earthquake, while 0.174% have suffered more than 11 earthquakes.

In [ ]:
bins = [0,1, 10, float('inf')]
labels = ['1','2-10', '11+']

binned = pd.cut(place_types, bins = bins, labels=labels)
binned_occurences = binned.value_counts().sort_index()


legend=['1 earthquake', '2-10 earthquakes', 'more than 11 earthquakes']
myexplode=[0, 0, 0.7]
binned_occurences.plot(kind='pie', labels=None, autopct='%1.3f%%')
plt.title('Distribution of frequency of earthquakes')
plt.legend(legend)
plt.show()

This brought us to our next question, are certain areas more prone to earthquakes with higher or lower frequency? We want to see the relationship and the ratio between the frequency of the earthquakes and the areas where these earthquakes happened. We will be using the Chi-Square Test for Independence.

In [ ]:
#place_types = earthquakes_df['place'].value_counts()
my_places = place_types
my_places.columns = ['place', 'count']
my_places = my_places.reset_index()

earthquake_dataframe = pd.merge(
    place_types,
    earthquakes_df[['place', 'latitude', 'longitude']],
    on='place',
    how='inner'
)

#this is variable 1
bins = [1, 5, 10, float('inf')]
labels = ['1 earthquake', 'between 5 and 10 earthquakes', 'more than 10 earthquakes']
earthquake_dataframe['count category'] = pd.cut(earthquake_dataframe['count'], bins = bins, labels=labels)

#this is variable 2
bins = 3
labels = ['South', 'Middle', 'North']
earthquake_dataframe['region category'] = pd.cut(earthquake_dataframe['latitude'], bins = bins, labels=labels)


#i need contingency table
contingency_table = pd.crosstab(earthquake_dataframe['count category'], earthquake_dataframe['region category'])


chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(contingency_table)

#print("Chi-square:", chi2)
#print("p-value:", p)

From the table we can see that locations with only one earthquake are predominantly in the South. The earthquake frequency is not evenly distributed across South, Middle and North.

Graph plotted below for Method 2: Based on magnitude and date.

Earlier we found out that regions in the southern latitude divide contain higher frequency earthquakes. We will now examine whether mag and depth are correlated using that region.

edit this: Now looking at the data below looks superoverwhelming but thats because location source directly from the USGS center is a lot but is can be seen that over the years USGS has recorded less earthquakes with a magnitude less than 4 and has recorded more earthquakes with a magnitude over 7 past 2011. This shows that over time the strength of the earthquakes have increased. From 2009 you can start to see less earthquakes under 4 with more occurances of earthquakes with mangnitude between 5 and 7.

In [ ]:
south_locations = earthquake_dataframe[earthquake_dataframe['region category'] == 'South']['place']

region = earthquakes_df[earthquakes_df['place'].isin(south_locations)].copy()

rvalue = region['mag'].corr(region['depth'])
print(f"Pearson correlation (Magnitude vs. Depth): {rvalue}")

plt.figure(figsize=(8, 10))
plt.scatter(region['mag'], region['depth'])
plt.title("Earthquake Magnitude vs. Depth (South Latitude Group)")
plt.xlabel("Magnitude")
plt.ylabel("Depth (km)")
plt.show()



There is a weak negative correlation but because there is so much data, we need to break some of this information down. We are going to still look at the south group. But we are going to visualize how many earthquakes there are by date.

In [ ]:
region['date'] = pd.to_datetime(region['date'])

region['date'].value_counts().sort_index().plot()
plt.xlabel("Date")
plt.ylabel("Earthquakes Occured")
plt.title("Earthquakes occured on each day recorded by USGS center")
plt.show()

We can see a huge spikes of earthquakes in around 2008, 2010, and 2013. But we are going to break this down a little further, and see the break down in just 2010 and see if there is more of a corrleation between Depth and magnitude since there will be less data points.

In [ ]:
region['year'] = pd.to_datetime(region['date']).dt.year
south_earthquakes = region.groupby(region['year'])
year = south_earthquakes.get_group(2010)

#then find magnitude v depth
rvalue2010 = year['depth'].corr(year['mag'])
print(f"Pearson r value for 2010 for depth and mag is: {rvalue2010}")
plt.figure(figsize=(8, 10))
plt.scatter(year['mag'], year['depth'])
plt.title("Earthquake Magnitude v. Depth in 2010")
plt.show()

edit: There is still a lot of data points and the R-value is still small but looking at the second graph it can be seen in 2010 that the larger than 5.5 that a negative corrleation to the depth of the earthquake.

**NOTE FOR GROUP MATES:**
what I did is I switched the methods to make it flow a little better. So Ana's analysis was about frequency in each region, and it concluded that the South divide of the globe contained a higher freq of earthquakes. i didn't edit her code. in order to make it flow like a story, i edited Natalie's method to examine the correlation of mag vs depth in that Southern region that had a higher frequency. I'm not sure how effective this is, i just wanted to relate the themes of our methods together. I wonder if in Natalie's method, we could also add a graph showing the correlation of mag vs depth using the original earthquakes df filtered to the year 2010? 